# 62) Z Oran Testi (Z Proportion Test)
Şimdiye kadarki testler (Z, T) hep **ortalamaları** (sayısal değişkenleri) karşılaştırıyordu. Z Oran Testi ise **oranları/yüzdeleri** karşılaştırır — "dönüşüm oranı," "tıklama oranı," "churn oranı" gibi **kategorik/ikili (başarı-başarısızlık) sonuçların** oranlarını test eder.

## Ne Zaman Kullanılır?
Değişkenin kendisi **Bernoulli tipinde** (sadece 2 sonuçlu: tıkladı/tıklamadı, satın aldı/almadı) olduğunda. A/B testlerinde en sık karşılaşacağımız test türlerinden biri budur.

## İki Versiyonu Var

**1. Tek Örneklem Oran Testi:** Bir oranın, iddia edilen bir değere eşit olup olmadığını test eder.
- H0: $p = p_0$ (örn: "dönüşüm oranımız %5'tir" iddiası)

**2. İki Örneklem Oran Testi (A/B Testinde En Sık Kullanılan):** İki grubun oranlarını karşılaştırır.
- H0: $p_1 = p_2$ (örn: "A ve B varyantının dönüşüm oranı aynıdır")

## Test İstatistiği Formülü (İki Örneklem İçin)
$$Z = \frac{\hat{p}_1 - \hat{p}_2}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_1}+\frac{1}{n_2}\right)}}$$

Burada $\hat{p}_1, \hat{p}_2$ iki grubun gözlenen oranları, $\hat{p}$ ise **birleşik (pooled) oran**. iki grubun toplam başarı sayısının, toplam gözlem sayısına bölünmesiyle bulunur:
$$\hat{p} = \frac{x_1 + x_2}{n_1 + n_2}$$

## Neden Z Kullanılır, T Değil?
Oran testlerinde CLT, Binom dağılımının normale yakınsamasına dayanır bu yüzden n yeterince büyükse (genelde $np \ge 5$ ve $n(1-p) \ge 5$ koşulu aranır) Z testi kullanılır, T testine gerek kalmaz.

## Python'da Kullanımı
`statsmodels.stats.proportion.proportions_ztest()` fonksiyonu kullanılır 
— `scipy`de doğrudan bir fonksiyon yoktur, bu yüzden `statsmodels`'a 
geçiyoruz.

## Örnek Senaryo (A/B Testi)
Bir e-ticaret sitesi, iki farklı buton rengini (A: kırmızı, B: yeşil) test ediyor:
- **A varyantı:** 1000 ziyaretçiden 120'si satın aldı
- **B varyantı:** 1000 ziyaretçiden 145'i satın aldı

B varyantının dönüşüm oranı gerçekten daha yüksek mi, yoksa bu fark şans eseri mi?

In [1]:
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

# --- Senaryo ---
# A varyantı: 1000 ziyaretçiden 120'si satın aldı
# B varyantı: 1000 ziyaretçiden 145'i satın aldı

# H0: A ve B varyantlarının dönüşüm oranları arasında fark yoktur (p_A = p_B)
# H1: A ve B varyantlarının dönüşüm oranları arasında fark vardır (p_A != p_B)

basari_sayilari = np.array([120, 145])   # her varyanttan kaç kişi satın aldı
gozlem_sayilari = np.array([1000, 1000])  # her varyantta kaç kişi vardı

# Dönüşüm oranlarına önce göz atalım
print(f"A dönüşüm oranı: {basari_sayilari[0]/gozlem_sayilari[0]:.4f}")
print(f"B dönüşüm oranı: {basari_sayilari[1]/gozlem_sayilari[1]:.4f}")

# Z Oran Testi
z_istatistigi, p_degeri = proportions_ztest(
    count=basari_sayilari, 
    nobs=gozlem_sayilari, 
    alternative='two-sided'
)

print(f"\nZ istatistiği: {z_istatistigi}")
print(f"p-değeri: {p_degeri}")

alpha = 0.05
if p_degeri < alpha:
    print("H0 reddedilir, iki varyant arasında anlamlı bir fark vardır.")
else:
    print("H0 reddedilemedi, iki varyant arasında anlamlı bir fark olduğuna dair yeterli kanıt yoktur.")

A dönüşüm oranı: 0.1200
B dönüşüm oranı: 0.1450

Z istatistiği: -1.648854485267929
p-değeri: 0.09917744952840564
H0 reddedilemedi, iki varyant arasında anlamlı bir fark olduğuna dair yeterli kanıt yoktur.


### Sonuç
Her iki varyantın dönüşüm oranı arasında anlamlı bir fark olup olmadığına yönelik yaptığımız z-oran testi sonuçlarına göre, gözlemlenen fark (%12 vs %14.5) istatistiksel olarak anlamlı bulunmamıştır (p>0.05). Bu doğrultuda, şu anki örneklem büyüklüğüyle B varyantının A'dan gerçekten daha iyi performans gösterdiğini iddia etmek için yeterli kanıt yoktur. Testin daha büyük bir örneklemle veya daha uzun süre çalıştırılması önerilebilir.